In [21]:
import geopandas as gpd
import glob
import os
import pandas as pd
import subprocess
import rasterio
import rioxarray
from whitebox import WhiteboxTools

import sys

# use absolute path here
project_path = "/mnt/School/PhD/AI221/Project/"
sys.path.insert(0, project_path)

from src.data_extraction.utils.constants import data_path

In [2]:
def generate_city_twi(
    psgc_code: int, 
    psgc_gdf: gpd.GeoDataFrame, 
    cop_dir: str,
    master_vrt: str, 
    output_dir: str
):
    """
    End-to-End pipeline to generate Topographic Wetness Index (TWI) for a specific city.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # ---------------------------------------------------------
    # File Paths (Temporary and Final)
    # ---------------------------------------------------------
    psgc_str = str(psgc_code)
    temp_buffer_geojson = os.path.join(cop_dir, f"temp_{psgc_str}_buffer.geojson")
    temp_dem_tif = os.path.join(cop_dir, f"temp_{psgc_str}_dem.tif")
    temp_filled_tif = os.path.join(cop_dir, f"temp_{psgc_str}_filled.tif")
    temp_slope_tif = os.path.join(cop_dir, f"temp_{psgc_str}_slope.tif")
    temp_flow_tif = os.path.join(cop_dir, f"temp_{psgc_str}_flow.tif")
    temp_twi_tif = os.path.join(cop_dir, f"temp_{psgc_str}_twi.tif")
    
    final_output_tif = os.path.join(output_dir, f"{psgc_str}_final_twi.tif")

    print(f"=== Starting TWI Pipeline for {psgc_code} ===")

    # Force to WGS84 Degrees before doing any math
    if psgc_gdf.crs != "EPSG:4326":
        psgc_gdf = psgc_gdf.to_crs("EPSG:4326")

    buffered_gdf = psgc_gdf.copy()
    # Create a 0.05 degree buffer (~5.5km) to catch upstream flow
    buffered_gdf['geometry'] = buffered_gdf.geometry.buffer(0.05)

    # FIX: Extract the rectangular bounds of the buffered area
    # Returns (minx, miny, maxx, maxy)
    minx, miny, maxx, maxy = buffered_gdf.total_bounds
    # buffered_gdf.to_file(temp_buffer_geojson, driver="GeoJSON")
    # print(f"Saved to {temp_buffer_geojson}")

    # ---------------------------------------------------------
    # "Smart Clip" from the Master VRT
    # ---------------------------------------------------------
    print("1. Extracting buffered DEM from master VRT...")
    gdal_cmd = [
            "gdalwarp",
            "-overwrite",
            "-te", str(minx), str(miny), str(maxx), str(maxy), # The Bounding Box
            master_vrt,
            temp_dem_tif
    ]

    result = subprocess.run(gdal_cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"GDAL ERROR:\n{result.stderr}")
        raise RuntimeError("gdalwarp failed to extract the bounding box.")
    
    # ---------------------------------------------------------
    # WhiteboxTools Hydrology Math
    # ---------------------------------------------------------
    print("2. Running hydrological calculations (WhiteboxTools)...")
    wbt = WhiteboxTools()
    wbt.set_verbose_mode(False)

    # 3a. Fill Depressions
    wbt.fill_depressions(dem=temp_dem_tif, output=temp_filled_tif, fix_flats=True)
    # 3b. Calculate Slope
    wbt.slope(dem=temp_filled_tif, output=temp_slope_tif, units="degrees")
    # 3c. D-Infinity Flow Accumulation
    wbt.d_inf_flow_accumulation(i=temp_filled_tif, output=temp_flow_tif)
    # 3d. Topographic Wetness Index
    wbt.wetness_index(sca=temp_flow_tif, slope=temp_slope_tif, output=temp_twi_tif)

    # ---------------------------------------------------------
    # The Final Precision Trim
    # ---------------------------------------------------------
    print("3. Trimming buffer to exact city boundaries...")
    twi_raster = rioxarray.open_rasterio(temp_twi_tif)
    
    # Clip using the original, un-buffered geometry
    clipped_twi = twi_raster.rio.clip(psgc_gdf.geometry, psgc_gdf.crs, drop=True)
    clipped_twi.rio.to_raster(final_output_tif)
    
    print(f"Success! Final data saved to: {final_output_tif}")

    # ---------------------------------------------------------
    # Phase 5: Cleanup
    # ---------------------------------------------------------
    print("5. Cleaning up temporary files...")
    temp_files = [
        temp_buffer_geojson, temp_dem_tif, temp_filled_tif, 
        temp_slope_tif, temp_flow_tif, temp_twi_tif
    ]
    for f in temp_files:
        if os.path.exists(f):
            os.remove(f)

In [14]:
cop_path = os.path.join(data_path, "ph_cop30_tiles")
vrt_path = os.path.join(cop_path, "philippines_master.vrt")
output_path = os.path.join(cop_path, "twi_per_city")

In [15]:
ph_bounds = gpd.read_file(os.path.join(data_path,"ph_adm3_municities/PH_Adm3_MuniCities.shp.shp"))
ph_bounds = ph_bounds[ph_bounds["geo_level"] == "City"]
ph_bounds.head()

,adm1_psgc,adm2_psgc,adm3_psgc,adm3_en,geo_level,len_crs,area_crs,len_km,area_km2,geometry
4,100000000,102800000,102805000,City of Batac,City,66661,158252391,66,158.0,"POLYGON ((247341.309 2003933.537, 247293.327 2..."
11,100000000,102800000,102812000,City of Laoag,City,53964,110146974,53,110.0,"POLYGON ((248393.247 2016552.78, 248424.831 20..."
28,100000000,102900000,102906000,City of Candon,City,62247,77652664,62,77.0,"POLYGON ((230319.064 1907309.065, 230338.289 1..."
56,100000000,102900000,102934000,City of Vigan,City,25067,24485368,25,24.0,"POLYGON ((221597.447 1945779.016, 221718.087 1..."
70,100000000,103300000,103314000,City of San Fernando,City,54233,99006121,54,99.0,"POLYGON ((225191.401 1841887.86, 225339.01 184..."


In [14]:
for psgc in ph_bounds["adm3_psgc"].unique():
    psgc_gdf = ph_bounds[ph_bounds["adm3_psgc"] == psgc]
    generate_city_twi(
        psgc_code=psgc,
        psgc_gdf = psgc_gdf,
        master_vrt=vrt_path,
        output_dir=output_path,
        cop_dir=cop_path
    )

=== Starting TWI Pipeline for 102805000 ===
1. Extracting buffered DEM from master VRT...
2. Running hydrological calculations (WhiteboxTools)...


/tmp/ipykernel_11952/1665727414.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  buffered_gdf['geometry'] = buffered_gdf.geometry.buffer(0.05)


3. Trimming buffer to exact city boundaries...
Success! Final data saved to: /mnt/School/PhD/AI221/Project/src/data/ph_cop30_tiles/twi_per_city/102805000_final_twi.tif
5. Cleaning up temporary files...
=== Starting TWI Pipeline for 102812000 ===
1. Extracting buffered DEM from master VRT...
2. Running hydrological calculations (WhiteboxTools)...


/tmp/ipykernel_11952/1665727414.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  buffered_gdf['geometry'] = buffered_gdf.geometry.buffer(0.05)


3. Trimming buffer to exact city boundaries...
Success! Final data saved to: /mnt/School/PhD/AI221/Project/src/data/ph_cop30_tiles/twi_per_city/102812000_final_twi.tif
5. Cleaning up temporary files...
=== Starting TWI Pipeline for 102906000 ===
1. Extracting buffered DEM from master VRT...
2. Running hydrological calculations (WhiteboxTools)...


...

3. Trimming buffer to exact city boundaries...
Success! Final data saved to: /mnt/School/PhD/AI221/Project/src/data/ph_cop30_tiles/twi_per_city/1900702000_final_twi.tif
5. Cleaning up temporary files...
=== Starting TWI Pipeline for 1903617000 ===
1. Extracting buffered DEM from master VRT...
2. Running hydrological calculations (WhiteboxTools)...


/tmp/ipykernel_11952/1665727414.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  buffered_gdf['geometry'] = buffered_gdf.geometry.buffer(0.05)


3. Trimming buffer to exact city boundaries...
Success! Final data saved to: /mnt/School/PhD/AI221/Project/src/data/ph_cop30_tiles/twi_per_city/1903617000_final_twi.tif
5. Cleaning up temporary files...
=== Starting TWI Pipeline for 1908703000 ===
1. Extracting buffered DEM from master VRT...
2. Running hydrological calculations (WhiteboxTools)...


/tmp/ipykernel_11952/1665727414.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  buffered_gdf['geometry'] = buffered_gdf.geometry.buffer(0.05)


3. Trimming buffer to exact city boundaries...
Success! Final data saved to: /mnt/School/PhD/AI221/Project/src/data/ph_cop30_tiles/twi_per_city/1908703000_final_twi.tif
5. Cleaning up temporary files...


## Validation

In [22]:
# expected PSGCs from your city boundary file
expected_psgcs = set(ph_bounds["adm3_psgc"].astype(str).unique())

# output directory
out_dir = os.path.join(data_path, "ph_cop30_tiles/twi_per_city")
print(out_dir)

records = []

for psgc in sorted(expected_psgcs):
    tif_path = os.path.join(out_dir, f"{psgc}_final_twi.tif")

    rec = {
        "psgc": psgc,
        "exists": os.path.exists(tif_path),
        "opens": False,
        "width": None,
        "height": None,
        "crs": None,
        "nodata": None,
        "valid_pixel_count": None,
        "all_nodata": None,
        "error": None,
    }

    if not os.path.exists(tif_path):
        records.append(rec)
        continue

    try:
        with rasterio.open(tif_path) as src:
            rec["opens"] = True
            rec["width"] = src.width
            rec["height"] = src.height
            rec["crs"] = str(src.crs) if src.crs else None
            rec["nodata"] = src.nodata

            arr = src.read(1, masked=True)
            valid_pixel_count = int(np.ma.count(arr))

            rec["valid_pixel_count"] = valid_pixel_count
            rec["all_nodata"] = valid_pixel_count == 0

    except Exception as e:
        rec["error"] = str(e)

    records.append(rec)

validation_df = pd.DataFrame(records)

validation_df["nonzero_shape"] = (
    validation_df["width"].fillna(0).gt(0) & validation_df["height"].fillna(0).gt(0)
)

validation_df["passes_basic_validation"] = (
    validation_df["exists"]
    & validation_df["opens"]
    & validation_df["nonzero_shape"]
    & ~validation_df["all_nodata"].fillna(True)
)

summary = {
    "expected_outputs": len(expected_psgcs),
    "files_found": int(validation_df["exists"].sum()),
    "files_opened": int(validation_df["opens"].sum()),
    "basic_passes": int(validation_df["passes_basic_validation"].sum()),
    "missing_files": int((~validation_df["exists"]).sum()),
    "failed_open": int((validation_df["exists"] & ~validation_df["opens"]).sum()),
    "all_nodata": int(validation_df["all_nodata"].fillna(False).sum()),
}

print(summary)

# show only problematic outputs
problems_df = validation_df.loc[~validation_df["passes_basic_validation"]].copy()
problems_df = problems_df.sort_values(["exists", "opens", "all_nodata", "psgc"])

display(problems_df)

../../data/ph_cop30_tiles/twi_per_city
{'expected_outputs': 149, 'files_found': 149, 'files_opened': 149, 'basic_passes': 149, 'missing_files': 0, 'failed_open': 0, 'all_nodata': 0}


,psgc,exists,opens,width,height,crs,nodata,valid_pixel_count,all_nodata,error,nonzero_shape,passes_basic_validation


In [23]:
# Getting the list of PSGCs that have been processed
tif_list = [
    int(
        tif
        .split("twi_per_city/")[1] # extract the full filename
        .split("_final_twi.tif")[0] # extract the name only
    )
    for tif in glob.glob(os.path.join(output_path, "*.tif"))
]
tif_list[:10]

[1001312000,
 1001321000,
 1004209000,
 1004210000,
 1004215000,
 1004307000,
 1004308000,
 102805000,
 102812000,
 102906000]

In [25]:
# Validation: each PSGC has a corresponding TWI TIF
set(ph_bounds["adm3_psgc"].unique()) == set(tif_list)

True